# Singapore HDB Rental: Data Cleaning, EDA, and Feature Relevance

This notebook develops an evidence-based direction for the rental prediction project. It:

1. audits and cleans the core and auxiliary datasets;
2. visualizes target, time, property, location, and market distributions;
3. engineers block, spatial, macroeconomic, and strictly historical rental features;
4. evaluates feature groups on a **forward time holdout**; and
5. uses permutation importance and error slices to identify promising specialized directions.

> The feature-relevance results are diagnostic, not final Kaggle estimates. A forward split is used because the test set occurs later than most training data.

## 1. Setup and reproducibility

Run this notebook from the project root or from a `notebooks/` subdirectory. The code locates the external data bundle automatically; set `CS5228_DATA_ROOT` if your Google Drive copy is elsewhere.

In [ ]:
from pathlib import Path
import re
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titleweight'] = 'bold'
RANDOM_STATE = 42

def find_project_root(start=Path.cwd()):
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'src' / 'data_paths.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the project source files. Open the notebook inside the project folder.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_paths import resolve_data_root
DATA_DIR = resolve_data_root(PROJECT_ROOT)
AUX_DIR = DATA_DIR / 'auxiliary'
print(f'Project root: {PROJECT_ROOT}')
print(f'Data root: {DATA_DIR}')

## 2. Load all supplied data

In [ ]:
paths = {
    'train': DATA_DIR / 'train.csv',
    'test': DATA_DIR / 'test.csv',
    'hdb_block': AUX_DIR / 'sg-hdb-block.csv',
    'mrt': AUX_DIR / 'sg-mrt-stations.csv',
    'schools': AUX_DIR / 'sg-schools.csv',
    'malls': AUX_DIR / 'sg-shopping-malls.csv',
    'coe': AUX_DIR / 'sg-coe-prices.csv',
    'stocks': AUX_DIR / 'sg-stock-prices.csv',
}
raw = {name: pd.read_csv(path) for name, path in paths.items()}
train_raw, test_raw = raw['train'], raw['test']

inventory = pd.DataFrame({
    'dataset': raw.keys(),
    'rows': [len(df) for df in raw.values()],
    'columns': [df.shape[1] for df in raw.values()],
    'missing_cells': [int(df.isna().sum().sum()) for df in raw.values()],
    'duplicate_rows': [int(df.duplicated().sum()) for df in raw.values()],
}).sort_values('rows', ascending=False)
display(inventory)
display(train_raw.head())

## 3. Cleaning utilities and quality audit

Cleaning decisions made here:

- normalize text case, whitespace, hyphens, and common street abbreviations;
- parse rental dates and numeric columns;
- normalize flat-type labels such as `4 room` and `4-room`;
- treat `-1` in HDB year/floor fields as missing;
- retain legitimate repeated rental approvals rather than deleting them blindly;
- flag constant columns instead of silently using them.

In [ ]:
STREET_REPLACEMENTS = {
    r'\bavenue\b': 'ave', r'\broad\b': 'rd', r'\bstreet\b': 'st',
    r'\bdrive\b': 'dr', r'\blane\b': 'lane', r'\bcentral\b': 'ctrl',
    r'\bplace\b': 'pl', r'\bclose\b': 'cl', r'\bcrescent\b': 'cres',
}

def normalize_text(series):
    return (series.astype('string').str.strip().str.lower()
            .str.replace(r'[-_/]+', ' ', regex=True)
            .str.replace(r'\s+', ' ', regex=True))

def normalize_street(series):
    out = normalize_text(series)
    for pattern, replacement in STREET_REPLACEMENTS.items():
        out = out.str.replace(pattern, replacement, regex=True)
    return out.str.replace(r'\s+', ' ', regex=True).str.strip()

def normalize_flat_type(series):
    out = normalize_text(series)
    out = out.str.replace(r'^(\d) room$', r'\1-room', regex=True)
    return out.replace({'executive': 'executive', 'multi generation': 'multi-generation'})

def clean_core(df, has_target):
    out = df.copy()
    out['RENT_APPROVAL_DATE'] = pd.to_datetime(out['RENT_APPROVAL_DATE'], format='%Y-%m', errors='coerce')
    for col in ['TOWN', 'BLOCK', 'FLAT_MODEL', 'FURNISHED']:
        out[col] = normalize_text(out[col])
    out['STREET'] = normalize_street(out['STREET'])
    out['FLAT_TYPE'] = normalize_flat_type(out['FLAT_TYPE'])
    numeric = ['FLOOR_AREA_SQM', 'LEASE_COMMENCE_DATE', 'FEE'] + (['MONTHLY_RENT'] if has_target else [])
    for col in numeric:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out['RENT_YEAR'] = out['RENT_APPROVAL_DATE'].dt.year
    out['RENT_MONTH'] = out['RENT_APPROVAL_DATE'].dt.month
    out['MONTH_INDEX'] = ((out['RENT_YEAR'] - 2021) * 12 + out['RENT_MONTH'] - 1).astype('Int64')
    out['LEASE_AGE'] = out['RENT_YEAR'] - out['LEASE_COMMENCE_DATE']
    out['ADDRESS_KEY'] = out['TOWN'] + '|' + out['BLOCK'] + '|' + out['STREET']
    return out

train = clean_core(train_raw, has_target=True)
test = clean_core(test_raw, has_target=False)

def column_audit(df):
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing_n': df.isna().sum(),
        'missing_pct': 100 * df.isna().mean(),
        'unique_n': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique_n'], ascending=[False, True])

display(column_audit(train))
constant_columns = [c for c in train.columns if train[c].nunique(dropna=False) <= 1]
print('Constant columns:', constant_columns)
print('Train dates:', train.RENT_APPROVAL_DATE.min().date(), 'to', train.RENT_APPROVAL_DATE.max().date())
print('Test dates: ', test.RENT_APPROVAL_DATE.min().date(), 'to', test.RENT_APPROVAL_DATE.max().date())

In [ ]:
# Distribution shift and category consistency audit
audit_rows = []
for col in ['TOWN', 'FLAT_TYPE', 'FLAT_MODEL', 'BLOCK', 'STREET', 'ADDRESS_KEY']:
    train_values = set(train[col].dropna().unique())
    test_values = set(test[col].dropna().unique())
    audit_rows.append({
        'feature': col,
        'train_unique': len(train_values),
        'test_unique': len(test_values),
        'unseen_test_categories': len(test_values - train_values),
        'test_rows_seen_in_train_pct': 100 * test[col].isin(train_values).mean(),
    })
display(pd.DataFrame(audit_rows))

feature_cols = [c for c in train_raw.columns if c != 'MONTHLY_RENT']
same_x = train_raw.groupby(feature_cols, dropna=False)['MONTHLY_RENT'].agg(['size', 'nunique']).reset_index()
print(f'Exact duplicate feature groups: {(same_x["size"] > 1).sum():,}')
print(f'Groups with identical observed features but different rents: {(same_x["nunique"] > 1).sum():,}')
display(same_x.sort_values(['nunique', 'size'], ascending=False).head(10))

## 4. Core-data EDA

These plots focus on questions that can change the modeling direction: skew/outliers, temporal drift, geographical variation, nonlinear size effects, and train/test coverage.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(train['MONTHLY_RENT'], bins=50, kde=True, ax=axes[0], color='#2a6fbb')
axes[0].set(title='Monthly rent distribution', xlabel='Monthly rent (SGD)')
sns.boxplot(x=train['MONTHLY_RENT'], ax=axes[1], color='#80b1d3')
axes[1].set(title='Rent spread and potential extremes', xlabel='Monthly rent (SGD)')
plt.tight_layout(); plt.show()
display(train['MONTHLY_RENT'].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_frame().T)

In [ ]:
monthly = train.groupby('RENT_APPROVAL_DATE')['MONTHLY_RENT'].agg(['count', 'median', 'mean'])
fig, ax1 = plt.subplots(figsize=(13, 5))
ax1.plot(monthly.index, monthly['median'], marker='o', ms=3, label='Median rent', color='#d95f02')
ax1.plot(monthly.index, monthly['mean'], alpha=.7, label='Mean rent', color='#1b9e77')
ax1.set(ylabel='Monthly rent (SGD)', title='Rental market drift over time')
ax2 = ax1.twinx()
ax2.bar(monthly.index, monthly['count'], width=20, alpha=.15, color='grey', label='Approvals')
ax2.set_ylabel('Training observations')
ax1.legend(loc='upper left'); plt.tight_layout(); plt.show()

counts_train = train.groupby('RENT_APPROVAL_DATE').size().rename('train')
counts_test = test.groupby('RENT_APPROVAL_DATE').size().rename('test')
monthly_counts = pd.concat([counts_train, counts_test], axis=1).fillna(0)
monthly_counts.index = monthly_counts.index.strftime('%Y-%m')
monthly_counts.plot(kind='bar', figsize=(15, 4), width=.85)
plt.title('Train/test observations by rental month'); plt.ylabel('Rows'); plt.xlabel('Month')
plt.xticks(rotation=90); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
type_order = train.groupby('FLAT_TYPE')['MONTHLY_RENT'].median().sort_values().index
sns.boxplot(data=train, y='FLAT_TYPE', x='MONTHLY_RENT', order=type_order, showfliers=False, ax=axes[0])
axes[0].set(title='Rent by flat type', xlabel='Monthly rent (SGD)', ylabel='Flat type')
town_order = train.groupby('TOWN')['MONTHLY_RENT'].median().sort_values().index
sns.boxplot(data=train, y='TOWN', x='MONTHLY_RENT', order=town_order, showfliers=False, ax=axes[1])
axes[1].set(title='Rent by town', xlabel='Monthly rent (SGD)', ylabel='Town')
plt.tight_layout(); plt.show()

In [ ]:
plot_sample = train.sample(min(20_000, len(train)), random_state=RANDOM_STATE)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(data=plot_sample, x='FLOOR_AREA_SQM', y='MONTHLY_RENT', hue='FLAT_TYPE', alpha=.18, s=18, legend=False, ax=axes[0])
axes[0].set(title='Rent versus floor area', xlabel='Floor area (sqm)', ylabel='Monthly rent (SGD)')
sns.scatterplot(data=plot_sample, x='LEASE_AGE', y='MONTHLY_RENT', hue='TOWN', alpha=.12, s=18, legend=False, ax=axes[1])
axes[1].set(title='Rent versus lease age', xlabel='Age at rental approval', ylabel='Monthly rent (SGD)')
plt.tight_layout(); plt.show()

## 5. Clean and understand the auxiliary data

In [ ]:
hdb = raw['hdb_block'].copy()
for col in ['TOWN', 'BLOCK', 'SUBZONE', 'PLANNING_AREA', 'REGION']:
    hdb[col] = normalize_text(hdb[col])
hdb['STREET'] = normalize_street(hdb['STREET'])
for col in ['LATITUDE', 'LONGITUDE', 'MAX_FLOOR', 'YEAR_COMPLETED']:
    hdb[col] = pd.to_numeric(hdb[col], errors='coerce')
hdb[['MAX_FLOOR', 'YEAR_COMPLETED']] = hdb[['MAX_FLOOR', 'YEAR_COMPLETED']].replace(-1, np.nan)
hdb['ADDRESS_KEY'] = hdb['TOWN'] + '|' + hdb['BLOCK'] + '|' + hdb['STREET']

mrt = raw['mrt'].copy()
schools = raw['schools'].copy()
malls = raw['malls'].copy()
for df in [mrt, schools, malls]:
    df['LATITUDE'] = pd.to_numeric(df['LATITUDE'], errors='coerce')
    df['LONGITUDE'] = pd.to_numeric(df['LONGITUDE'], errors='coerce')
mrt['STATUS'] = normalize_text(mrt['STATUS'])
schools['MAINLEVEL_CODE'] = normalize_text(schools['MAINLEVEL_CODE'])

coe = raw['coe'].copy()
month_lookup = {m.lower(): i for i, m in enumerate(['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec'], 1)}
coe['MONTH_NUM'] = normalize_text(coe['MONTH']).map(month_lookup)
coe['DATE'] = pd.to_datetime(dict(year=pd.to_numeric(coe['YEAR']), month=coe['MONTH_NUM'], day=1))
for col in ['QUOTA_PREMIUM', 'PREVAILING_QUOTA_PREMIUM', 'QUOTA', 'BIDS_RECEIVED']:
    coe[col] = pd.to_numeric(coe[col].astype(str).str.replace(r'[$,]', '', regex=True), errors='coerce')
coe['BID_QUOTA_RATIO'] = coe['BIDS_RECEIVED'] / coe['QUOTA'].replace(0, np.nan)

stocks = raw['stocks'].copy()
stocks['DATE'] = pd.to_datetime(stocks['DATE'], errors='coerce')
for col in ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'ADJUSTED_CLOSE', 'VOLUME']:
    stocks[col] = pd.to_numeric(stocks[col], errors='coerce')

aux_summary = pd.DataFrame([
    {'dataset':'HDB blocks', 'rows':len(hdb), 'coverage_or_types':f'{hdb.TOWN.nunique()} towns', 'model_role':'building + precise geography'},
    {'dataset':'MRT', 'rows':len(mrt), 'coverage_or_types':mrt.STATUS.value_counts().to_dict(), 'model_role':'transport accessibility'},
    {'dataset':'Schools', 'rows':len(schools), 'coverage_or_types':schools.MAINLEVEL_CODE.value_counts().head().to_dict(), 'model_role':'family amenities'},
    {'dataset':'Malls', 'rows':len(malls), 'coverage_or_types':'point locations', 'model_role':'retail accessibility'},
    {'dataset':'COE', 'rows':len(coe), 'coverage_or_types':f'{coe.DATE.min():%Y-%m} to {coe.DATE.max():%Y-%m}', 'model_role':'lagged market proxy'},
    {'dataset':'Stocks', 'rows':len(stocks), 'coverage_or_types':f'{stocks.SYMBOL.nunique()} symbols; {stocks.DATE.min():%Y-%m} to {stocks.DATE.max():%Y-%m}', 'model_role':'lagged market proxy'},
])
display(aux_summary)
display(column_audit(hdb).head(12))

## 6. Join block attributes and engineer spatial accessibility

Distances are calculated once per HDB block with the haversine formula, then joined to rental rows. `NEAREST_OPEN_MRT_KM` is preferable to using planned stations as though they had always been available. Historical opening dates are not supplied, so even the current-open measure should be treated as a snapshot limitation.

In [ ]:
def minimum_haversine_km(source_lat, source_lon, target_lat, target_lon, batch_size=1000):
    source = np.radians(np.c_[source_lat, source_lon].astype(float))
    target = np.radians(np.c_[target_lat, target_lon].astype(float))
    result = np.full(len(source), np.nan)
    valid_targets = np.isfinite(target).all(axis=1)
    target = target[valid_targets]
    for start in range(0, len(source), batch_size):
        stop = min(start + batch_size, len(source))
        chunk = source[start:stop]
        valid_sources = np.isfinite(chunk).all(axis=1)
        if not valid_sources.any() or len(target) == 0:
            continue
        lat1 = chunk[valid_sources, 0][:, None]
        lon1 = chunk[valid_sources, 1][:, None]
        lat2 = target[:, 0][None, :]
        lon2 = target[:, 1][None, :]
        a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
        d = 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
        chunk_result = np.full(len(chunk), np.nan)
        chunk_result[valid_sources] = d.min(axis=1)
        result[start:stop] = chunk_result
    return result

block_features = hdb.drop_duplicates('ADDRESS_KEY').copy()
open_mrt = mrt[mrt['STATUS'].eq('open')]
primary = schools[schools['MAINLEVEL_CODE'].eq('primary')]
for name, places in {
    'NEAREST_OPEN_MRT_KM': open_mrt,
    'NEAREST_ANY_MRT_KM': mrt,
    'NEAREST_MALL_KM': malls,
    'NEAREST_PRIMARY_SCHOOL_KM': primary,
}.items():
    block_features[name] = minimum_haversine_km(
        block_features['LATITUDE'], block_features['LONGITUDE'],
        places['LATITUDE'], places['LONGITUDE'])

keep_block = ['ADDRESS_KEY','POSTAL_CODE','LATITUDE','LONGITUDE','MAX_FLOOR','YEAR_COMPLETED',
              'SUBZONE','PLANNING_AREA','REGION','NEAREST_OPEN_MRT_KM','NEAREST_ANY_MRT_KM',
              'NEAREST_MALL_KM','NEAREST_PRIMARY_SCHOOL_KM']
model_df = train.merge(block_features[keep_block], on='ADDRESS_KEY', how='left', validate='many_to_one')
model_df['BUILDING_AGE'] = model_df['RENT_YEAR'] - model_df['YEAR_COMPLETED']
print(f'HDB block join coverage: {100 * model_df.LATITUDE.notna().mean():.2f}%')
display(model_df[['LATITUDE','LONGITUDE','MAX_FLOOR','YEAR_COMPLETED','NEAREST_OPEN_MRT_KM',
                  'NEAREST_MALL_KM','NEAREST_PRIMARY_SCHOOL_KM']].describe().T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
geo_sample = model_df.sample(min(30_000, len(model_df)), random_state=RANDOM_STATE)
sc = axes[0].scatter(geo_sample['LONGITUDE'], geo_sample['LATITUDE'], c=geo_sample['MONTHLY_RENT'],
                     cmap='viridis', s=7, alpha=.35)
axes[0].set(title='Spatial distribution of observed rent', xlabel='Longitude', ylabel='Latitude')
plt.colorbar(sc, ax=axes[0], label='Monthly rent (SGD)')
distance_bins = pd.qcut(model_df['NEAREST_OPEN_MRT_KM'], q=10, duplicates='drop')
distance_effect = model_df.groupby(distance_bins, observed=True)['MONTHLY_RENT'].agg(['median','count'])
axes[1].plot(range(len(distance_effect)), distance_effect['median'], marker='o')
axes[1].set(title='Median rent by MRT-distance decile', xlabel='Nearest-MRT distance decile', ylabel='Median rent (SGD)')
axes[1].set_xticks(range(len(distance_effect))); axes[1].set_xticklabels([str(x) for x in distance_effect.index], rotation=45, ha='right')
plt.tight_layout(); plt.show()

## 7. Build lagged macroeconomic features

COE and stock variables are aggregated to month and shifted by one month. Lagging makes the diagnostic closer to a real forecasting setting: information from the rental month itself may not have been known when producing the prediction.

In [ ]:
coe_monthly = (coe.groupby('DATE')
               .agg(COE_PREMIUM_MEAN=('QUOTA_PREMIUM','mean'),
                    COE_BID_QUOTA_RATIO=('BID_QUOTA_RATIO','mean'))
               .sort_index().shift(1).add_suffix('_LAG1').reset_index())

stocks = stocks.sort_values(['SYMBOL','DATE'])
stocks['DAILY_RETURN'] = stocks.groupby('SYMBOL')['ADJUSTED_CLOSE'].pct_change(fill_method=None)
stocks['MONTH'] = stocks['DATE'].dt.to_period('M').dt.to_timestamp()
stock_monthly = (stocks.groupby('MONTH')
                 .agg(STOCK_MARKET_RETURN=('DAILY_RETURN','mean'),
                      STOCK_MARKET_VOLATILITY=('DAILY_RETURN','std'))
                 .sort_index().shift(1).add_suffix('_LAG1').reset_index()
                 .rename(columns={'MONTH':'DATE'}))

model_df = (model_df.merge(coe_monthly, left_on='RENT_APPROVAL_DATE', right_on='DATE', how='left')
            .drop(columns='DATE')
            .merge(stock_monthly, left_on='RENT_APPROVAL_DATE', right_on='DATE', how='left')
            .drop(columns='DATE'))
macro_cols = ['COE_PREMIUM_MEAN_LAG1','COE_BID_QUOTA_RATIO_LAG1',
              'STOCK_MARKET_RETURN_LAG1','STOCK_MARKET_VOLATILITY_LAG1']
display(model_df.groupby('RENT_APPROVAL_DATE')[macro_cols].first().tail(12))

## 8. Strictly historical address features

The same address occurs repeatedly, so historical rent may be more valuable than many external variables. To prevent target leakage, each address-month receives a median based only on **earlier months**. Rentals from the same month never supply one another's target.

In [ ]:
address_month = (train.groupby(['ADDRESS_KEY','RENT_APPROVAL_DATE'], as_index=False)
                 .agg(ADDRESS_MONTH_RENT=('MONTHLY_RENT','median'),
                      ADDRESS_MONTH_OBS=('MONTHLY_RENT','size'))
                 .sort_values(['ADDRESS_KEY','RENT_APPROVAL_DATE']))
address_month['ADDRESS_PRIOR_MONTHS'] = address_month.groupby('ADDRESS_KEY').cumcount()
address_month['ADDRESS_PRIOR_MEDIAN'] = (address_month.groupby('ADDRESS_KEY')['ADDRESS_MONTH_RENT']
                                          .transform(lambda s: s.shift().expanding().median()))
history_cols = ['ADDRESS_KEY','RENT_APPROVAL_DATE','ADDRESS_PRIOR_MONTHS','ADDRESS_PRIOR_MEDIAN']
model_df = model_df.merge(address_month[history_cols], on=['ADDRESS_KEY','RENT_APPROVAL_DATE'],
                          how='left', validate='many_to_one')
print(f'Rows with a strictly earlier address rent: {100 * model_df.ADDRESS_PRIOR_MEDIAN.notna().mean():.2f}%')
coverage_by_month = model_df.groupby('RENT_APPROVAL_DATE')['ADDRESS_PRIOR_MEDIAN'].apply(lambda s: s.notna().mean())
coverage_by_month.plot(figsize=(12,4), marker='o', ms=3)
plt.title('Availability of prior same-address rent over time'); plt.ylabel('Coverage'); plt.xlabel('Month')
plt.ylim(0, 1.05); plt.tight_layout(); plt.show()

## 9. Forward-holdout feature-group evaluation

The last 12 training months are held out. The diagnostic model is histogram gradient boosting with one-hot encoding for moderate-cardinality categories. `BLOCK` and `STREET` are intentionally excluded: the historical features represent address repetition more safely than arbitrary one-hot address memorization.

Interpret changes in validation MAE/RMSE—not training fit—as evidence of incremental relevance.

In [ ]:
TARGET = 'MONTHLY_RENT'
validation_start = model_df['RENT_APPROVAL_DATE'].max() - pd.DateOffset(months=11)
train_mask = model_df['RENT_APPROVAL_DATE'] < validation_start
valid_mask = ~train_mask
print(f'Train through {(validation_start - pd.DateOffset(months=1)):%Y-%m}: {train_mask.sum():,} rows')
print(f'Validate {validation_start:%Y-%m} to {model_df.RENT_APPROVAL_DATE.max():%Y-%m}: {valid_mask.sum():,} rows')

feature_groups = {
    'core': ['MONTH_INDEX','RENT_MONTH','FLOOR_AREA_SQM','LEASE_AGE','TOWN','FLAT_TYPE','FLAT_MODEL'],
    'block': ['MAX_FLOOR','YEAR_COMPLETED','BUILDING_AGE','LATITUDE','LONGITUDE','SUBZONE','PLANNING_AREA','REGION'],
    'spatial': ['NEAREST_OPEN_MRT_KM','NEAREST_MALL_KM','NEAREST_PRIMARY_SCHOOL_KM'],
    'macro_lagged': macro_cols,
    'address_history': ['ADDRESS_PRIOR_MONTHS','ADDRESS_PRIOR_MEDIAN'],
}

def make_pipeline(X):
    categorical = [c for c in X.columns if str(X[c].dtype) in ('object', 'string')]
    numeric = [c for c in X.columns if c not in categorical]
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                          ('scale', StandardScaler())]), numeric),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                          ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=10, sparse_output=False))]), categorical),
    ], remainder='drop')
    model = HistGradientBoostingRegressor(learning_rate=.08, max_iter=250, max_leaf_nodes=31,
                                          l2_regularization=2.0, random_state=RANDOM_STATE)
    return Pipeline([('prepare', preprocessor), ('model', model)])

def score_features(features):
    X = model_df[features].copy()
    y = model_df[TARGET]
    pipeline = make_pipeline(X)
    pipeline.fit(X.loc[train_mask], y.loc[train_mask])
    pred = pipeline.predict(X.loc[valid_mask])
    return pipeline, pred, {
        'MAE': mean_absolute_error(y.loc[valid_mask], pred),
        'RMSE': mean_squared_error(y.loc[valid_mask], pred) ** 0.5,
        'R2': r2_score(y.loc[valid_mask], pred),
        'n_features_before_encoding': len(features),
    }

experiment_sets = {
    'Core only': feature_groups['core'],
    '+ Block': feature_groups['core'] + feature_groups['block'],
    '+ Spatial': feature_groups['core'] + feature_groups['block'] + feature_groups['spatial'],
    '+ Macro': feature_groups['core'] + feature_groups['block'] + feature_groups['spatial'] + feature_groups['macro_lagged'],
    '+ Address history': sum(feature_groups.values(), []),
}
results, fitted = [], {}
for name, features in experiment_sets.items():
    pipe, pred, metrics = score_features(features)
    results.append({'experiment': name, **metrics})
    fitted[name] = {'pipeline': pipe, 'prediction': pred, 'features': features}
    print(f'{name:>18}: MAE={metrics["MAE"]:,.1f}, RMSE={metrics["RMSE"]:,.1f}, R2={metrics["R2"]:.3f}')
results_df = pd.DataFrame(results).set_index('experiment')
results_df['MAE_improvement_vs_core_pct'] = 100 * (results_df.loc['Core only','MAE'] - results_df['MAE']) / results_df.loc['Core only','MAE']
display(results_df)
results_df[['MAE','RMSE']].plot(kind='bar', figsize=(11,5), rot=20)
plt.title('Incremental value of auxiliary feature groups on future months'); plt.ylabel('Error (SGD)')
plt.tight_layout(); plt.show()

## 10. Permutation feature importance

Permutation importance measures how much validation error worsens when one original feature is shuffled. It captures the fitted model's reliance on a feature, not causality. Correlated features may share or conceal importance.

In [ ]:
final_run = fitted['+ Address history']
X_valid = model_df.loc[valid_mask, final_run['features']]
y_valid = model_df.loc[valid_mask, TARGET]
importance_sample = X_valid.sample(min(12_000, len(X_valid)), random_state=RANDOM_STATE)
importance_y = y_valid.loc[importance_sample.index]
perm = permutation_importance(
    final_run['pipeline'], importance_sample, importance_y,
    scoring='neg_mean_absolute_error', n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
importance_df = (pd.DataFrame({
    'feature': final_run['features'],
    'mae_increase_mean': perm.importances_mean,
    'mae_increase_std': perm.importances_std,
}).sort_values('mae_increase_mean', ascending=False))
display(importance_df)
top = importance_df.head(20).sort_values('mae_increase_mean')
plt.figure(figsize=(10, 7))
plt.barh(top['feature'], top['mae_increase_mean'], xerr=top['mae_increase_std'], color='#4c78a8', alpha=.85)
plt.axvline(0, color='black', lw=1)
plt.title('Permutation importance on future-month holdout')
plt.xlabel('Increase in MAE after shuffling feature (larger = more useful)')
plt.tight_layout(); plt.show()

## 11. Error slices and specialized-direction diagnostics

Aggregate accuracy can hide systematic weaknesses. These slices test whether the next modeling effort should emphasize temporal forecasting, address history, spatial accessibility, or specific flat/town segments.

In [ ]:
validation = model_df.loc[valid_mask, ['RENT_APPROVAL_DATE','TOWN','FLAT_TYPE','MONTHLY_RENT',
                                        'ADDRESS_PRIOR_MEDIAN','ADDRESS_PRIOR_MONTHS','NEAREST_OPEN_MRT_KM']].copy()
validation['PREDICTED'] = final_run['prediction']
validation['ABS_ERROR'] = (validation['MONTHLY_RENT'] - validation['PREDICTED']).abs()
validation['HAS_ADDRESS_HISTORY'] = validation['ADDRESS_PRIOR_MEDIAN'].notna()
validation['MRT_DISTANCE_BAND'] = pd.cut(validation['NEAREST_OPEN_MRT_KM'],
                                         bins=[-np.inf,.4,.8,1.2,2,np.inf],
                                         labels=['<=0.4 km','0.4-0.8','0.8-1.2','1.2-2','>2 km'])

def error_slice(column, min_rows=100):
    out = validation.groupby(column, observed=True).agg(
        rows=('ABS_ERROR','size'), MAE=('ABS_ERROR','mean'),
        median_actual=('MONTHLY_RENT','median'), median_predicted=('PREDICTED','median'))
    return out[out['rows'] >= min_rows].sort_values('MAE', ascending=False)

print('Error with versus without earlier same-address evidence')
display(error_slice('HAS_ADDRESS_HISTORY', min_rows=1))
print('Highest-error flat types')
display(error_slice('FLAT_TYPE'))
print('Highest-error towns')
display(error_slice('TOWN').head(12))
print('Error by MRT distance')
display(error_slice('MRT_DISTANCE_BAND'))

monthly_error = validation.groupby('RENT_APPROVAL_DATE')['ABS_ERROR'].mean()
monthly_error.plot(marker='o', figsize=(12,4))
plt.title('Validation MAE by forecast month'); plt.ylabel('MAE (SGD)'); plt.xlabel('Month')
plt.tight_layout(); plt.show()

In [ ]:
# Automated evidence summary: use this as a starting point for team discussion, not as a final conclusion.
core_mae = results_df.loc['Core only', 'MAE']
best_name = results_df['MAE'].idxmin()
best_mae = results_df.loc[best_name, 'MAE']
group_gains = results_df['MAE'].shift(1) - results_df['MAE']
history_coverage = model_df.loc[valid_mask, 'ADDRESS_PRIOR_MEDIAN'].notna().mean()
rent_growth = (monthly['median'].iloc[-1] / monthly['median'].iloc[0] - 1)

print('EVIDENCE-BASED DIRECTION')
print('-' * 70)
print(f'Best diagnostic specification: {best_name} (MAE {best_mae:,.1f} SGD).')
print(f'Improvement over core-only diagnostic: {core_mae - best_mae:,.1f} SGD ({100*(core_mae-best_mae)/core_mae:.1f}%).')
print(f'Strict prior-address coverage in validation: {history_coverage:.1%}.')
print(f'Change in monthly median rent from first to last training month: {rent_growth:.1%}.')
print()
if group_gains.get('+ Address history', 0) > 10 and history_coverage > .5:
    print('→ Prioritize leakage-safe block/address rent dynamics and recency features.')
if group_gains.get('+ Spatial', 0) > 5:
    print('→ Spatial accessibility adds signal; refine MRT opening dates, station exits, and travel-time features.')
if group_gains.get('+ Macro', 0) <= 2:
    print('→ Lagged macro variables add little here; keep them low priority unless rolling folds disagree.')
else:
    print('→ Macro variables show incremental signal; verify it across several rolling time folds.')
if abs(rent_growth) > .1:
    print('→ Temporal drift is material; use rolling-origin validation and explicit market-trend features.')
print('→ Inspect the preceding error tables to select targeted flat types/towns for interaction features.')

## 12. Where to go next

Use the evidence above to choose a specialized path:

- **Address-history path:** add recency, last observed rent, exponentially weighted history, and town-level market adjustment. All target-derived features must use earlier months only.
- **Spatial path:** add MRT opening dates, station exits, interchange status, bus connectivity, amenity counts within radii, and travel time to employment centres.
- **Temporal path:** develop town/flat-type rental indices, rolling-origin folds, lagged market variables, and models robust to longer forecast horizons.
- **Segment path:** model distinct flat types or regions separately only when error slices show stable, sizeable differences.

Before final model selection, repeat the ablation study across multiple rolling-origin folds and compare CatBoost with LightGBM/XGBoost. The present model is intentionally a lightweight diagnostic.